# MacroTrader Model Pipeline - 08/09/2024

## About

This NB presents an end-to-end pipeline for our Macrotrader Layer

### Explanation of Key Components

1. **Initialization (`__init__`)**:
   - Initializes the main parameters for the trading model, including the ticker symbol, date range, timeframe, and inventory. This sets up the environment for the model's data processing and execution.

2. **Data Pulling (`pull_data`)**:
   - Uses the Polygon.io API to pull OHLCV (Open, High, Low, Close, Volume) and bid-ask data, merging them into a single DataFrame. This data serves as the foundation for the model's predictions.

3. **Technical Indicators (`add_technical_indicators`)**:
   - Adds a variety of technical indicators (e.g., RSI, MACD, Bollinger Bands) to the data, providing essential features for trading signals.

4. **Forecasting (`add_forecasts`, `add_real_time_forecasts`)**:
   - Generates forecasts for key indicators using models like ARIMA and Exponential Smoothing. These forecasts are used to predict future market conditions.

5. **Custom Trading Environment (`CustomTradingEnvironment`)**:
   - Defines a custom Gym environment for simulating trades based on the model’s predictions. It manages the state, action space, and trade execution logic.

6. **Meta-Learner (`MetaLearner`)**:
   - Classifies the trading scenario based on transaction size and timeframe. This helps in selecting the appropriate pre-trained model for the given trading conditions.

7. **Model Loading (`Model`)**:
   - Loads pre-trained models for different trading scenarios. The models are configured with a custom U-Net Transformer for feature extraction and use Proximal Policy Optimization (PPO) for trading decisions.

8. **Policy and Model (`CustomTransformerPolicy`, `CustomUNetTransformerModel`)**:
   - Defines a custom policy using a U-Net Transformer model for feature extraction, which is integrated into the PPO framework for decision-making.

9. **Pipeline Execution (`run_pipeline`)**:
   - Runs the entire trading pipeline, from data retrieval and indicator addition to forecasting and generating trade schedules. It orchestrates the entire process to produce trading decisions.

### Usage Directions

1. **Setup the Trading Model**:
   - Initialize the `TradingModel` class with the required parameters like `ticker`, `start_date`, `end_date`, `timeframe`, and `inventory`.

   ```python
   model = TradingModel(ticker="AAPL", start_date="2023-01-01", end_date="2023-01-31", timeframe=390, inventory=1000)
    ```
    
2. **Run the Pipeline**:
   - Execute the `run_pipeline` method to pull data, add technical indicators, generate forecasts, and produce a trading schedule.

   ```python
   trades = model.run_pipeline()
    ```
    
3. **Review Trades**:
   - The `trades` variable will contain the list of trades executed by the model. You can print or analyze these trades


### Installation Instructions

If you need to ensure compatibility with specific versions and additional dependencies, use the following installation commands:

```bash
# Install required packages
!pip install gym==0.26.0 shimmy>=0.2.1 stable-baselines3 torch

# Install additional required packages
!pip install h5py
!pip install arch

# Install TA-Lib using conda for technical analysis indicators
!conda install -c conda-forge ta-lib -y
```

These commands will set up your environment with the necessary dependencies for the trading model.

In [1]:
# !pip uninstall gymnasium shimmy stable-baselines3 -y
!pip install gym==0.26.0 shimmy>=0.2.1 stable-baselines3 torch
!pip install h5py
!pip install arch
!conda install -c conda-forge ta-lib -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 84.7 MB/s eta 0:00:00ta 0:00:01
done
Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 23.3.1
  latest version: 24.7.1

Please update conda by running

    $ conda update -n base -c conda-forge conda

Or to minimize the number of packages updated during conda update use

     conda install conda=24.7.1



## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/pytorch_p310

  added / updated specs:
    - ta-lib


The following packages will be UPDATED:

  certifi                             2024.2.2-pyhd8ed1ab_0 --> 2024.7.4-pyhd8ed1ab_0 




Preparing transaction: done
Verifying transaction: done
Executing transaction: done


In [62]:
import requests
import time
import gym
import csv
import pandas as pd
import numpy as np
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3 import PPO
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from gym import spaces
from datetime import datetime, timedelta
from pandas.tseries.holiday import USFederalHolidayCalendar
from arch import arch_model


import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")


class TradingModel:
    def __init__(self, ticker, start_date, end_date, timeframe, inventory):
        """
        Initializes the TradingModel class with the given parameters.

        Args:
        - ticker (str): The stock ticker symbol (e.g., 'AAPL').
        - start_date (str): The start date for data retrieval in 'YYYY-MM-DD' format.
        - end_date (str): The end date for data retrieval in 'YYYY-MM-DD' format.
        - timeframe (int): The timeframe for trade execution.
        - inventory (int): The initial inventory of shares to trade.
        """
        self.ticker = ticker
        self.start_date = start_date
        self.end_date = end_date
        self.timeframe = timeframe
        self.inventory = inventory
        
    def pull_data(self):
        """
        Pulls OHLCV and Bid-Ask data for the specified ticker and date range, and merges them.

        Returns:
        - pd.DataFrame: The merged DataFrame containing OHLCV and Bid-Ask data.
        """
        print("LOGGING: Pulling Data...")

        def fetch_and_merge_data(ticker, start_date, end_date):
            api_key = 'r65B9O5aplSJn7BWSo8z8pNH8v2wW5yc' 

            def get_ticker_agg_data(start_date, end_date, ticker, api_key):
                try:
                    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/minute/{start_date}/{end_date}"
                    params = {"apiKey": api_key, "limit": 50000}
                    response = requests.get(url, params=params)
                    response.raise_for_status()
                    data = response.json()

                    if 'results' not in data:
                        raise ValueError("No results found in the API response.")

                    # Save OHLCV data to CSV
                    with open(f'data_{ticker}_{start_date}_{end_date}.csv', 'w', newline='') as csvfile:
                        fieldnames = ['timestamp', 'datetime', 'open', 'high', 'low', 'close', 'volume']
                        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                        writer.writeheader()
                        for item in data['results']:
                            human_readable_time = pd.to_datetime(item['t'], unit='ms')
                            writer.writerow({
                                'timestamp': item['t'],
                                'datetime': human_readable_time,
                                'open': item['o'],
                                'high': item['h'],
                                'low': item['l'],
                                'close': item['c'],
                                'volume': item['v']
                            })
                    return data

                except requests.exceptions.HTTPError as http_err:
                    print(f"HTTP error occurred: {http_err}")
                except requests.exceptions.ConnectionError as conn_err:
                    print(f"Connection error occurred: {conn_err}")
                except requests.exceptions.Timeout as timeout_err:
                    print(f"Timeout error occurred: {timeout_err}")
                except requests.exceptions.RequestException as req_err:
                    print(f"An error occurred while making the request: {req_err}")
                except ValueError as val_err:
                    print(f"Value error: {val_err}")
                except Exception as e:
                    print(f"An error occurred: {e}")

                return None

            def get_hourly_quotes(date, ticker, api_key):
                try:
                    today = pd.to_datetime(date)
                    quotes = []
                    for n in range(23):
                        gt = (today + pd.DateOffset(hours=n)).value
                        lt = (today + pd.DateOffset(hours=n+1)).value

                        response = requests.get(
                            f'https://api.polygon.io/v3/quotes/{ticker}', 
                            params={"apiKey": api_key, "timestamp.gt": gt, "timestamp.lt": lt, "limit": 1000}
                        )
                        response.raise_for_status()
                        data = response.json()

                        if response.status_code == 200 and 'results' in data:
                            quotes.extend(data['results'])
                        else:
                            print(f"No quotes found for {date}, hour {n}")

                    return quotes

                except requests.exceptions.HTTPError as http_err:
                    print(f"HTTP error occurred: {http_err}")
                except requests.exceptions.ConnectionError as conn_err:
                    print(f"Connection error occurred: {conn_err}")
                except requests.exceptions.Timeout as timeout_err:
                    print(f"Timeout error occurred: {timeout_err}")
                except requests.exceptions.RequestException as req_err:
                    print(f"An error occurred while making the request: {req_err}")
                except Exception as e:
                    print(f"An error occurred: {e}")

                return []

            def get_ticker_quote_data(start_date, end_date, ticker, api_key):
                try:
                    start = pd.Timestamp(start_date)
                    end = pd.Timestamp(end_date)
                    date_range = pd.date_range(start=start, end=end, freq='D')
                    frames = []

                    for date in date_range.strftime('%Y-%m-%d').tolist():
                        quotes = get_hourly_quotes(date, ticker, api_key)
                        if quotes:
                            df = pd.DataFrame(quotes)
                            frames.append(df)

                    if frames:
                        df = pd.concat(frames)
                        df["datetime"] = pd.to_datetime(df["participant_timestamp"], unit="ns")
                        df = df.set_index('datetime')
                        df.drop(
                            columns=["indicators", "ask_exchange", "bid_exchange",  
                                     "participant_timestamp", "tape", "sip_timestamp", 
                                     "conditions", "sequence_number"],
                            inplace=True
                        )

                        quotes = df.resample('min').agg({
                            'ask_price': 'last',
                            'ask_size': 'sum',
                            'bid_price': 'last',
                            'bid_size': 'sum',
                        }).fillna(method='ffill')

                        return quotes

                    else:
                        print(f"No data frames available for the given date range: {start_date} to {end_date}")
                        return pd.DataFrame()

                except Exception as e:
                    print(f"An error occurred while fetching quote data: {e}")
                    return pd.DataFrame()

            # Fetch OHLCV Agg Data
            ohlcv = get_ticker_agg_data(start_date, end_date, ticker, api_key=api_key)
            if ohlcv is None:
                print("Failed to retrieve OHLCV data.")
                return pd.DataFrame()

            # Load OHLCV data
            try:
                ohlcv = pd.read_csv(f"data_{ticker}_{start_date}_{end_date}.csv")
            except Exception as e:
                print(f"Error loading OHLCV data from CSV: {e}")
                return pd.DataFrame()

            # Fetch Bid Ask Data over the same period
            quotes = get_ticker_quote_data(start_date, end_date, ticker, api_key=api_key)
            if quotes.empty:
                print("Failed to retrieve Bid-Ask quote data.")
                return pd.DataFrame()

            try:
                quotes["timestamp"] = quotes.index.astype(int) // (10**6)  # convert to ms

                # Merge and process output
                result = pd.merge(ohlcv, quotes, left_on="timestamp", right_on="timestamp", how="inner")

                # Write combined file
                result.to_csv(f"merged_data_{ticker}_{start_date}_{end_date}.csv")
                print(f"Merged data saved to merged_data_{ticker}_{start_date}_{end_date}.csv")

                return result

            except Exception as e:
                print(f"An error occurred during merging or saving the data: {e}")
                return pd.DataFrame()

        # Call the function to fetch and merge data
        df = fetch_and_merge_data(self.ticker, self.start_date, self.end_date)
        return df
    
    def add_technical_indicators(self, data):
        """
        Adds various technical indicators to the data.

        Args:
        - data (pd.DataFrame): The DataFrame containing OHLCV and Bid-Ask data.

        Returns:
        - pd.DataFrame: The DataFrame with additional technical indicators.
        """
        print("LOGGING: Adding Technical Indicators to data...")
        
        def add_momentum_indicators(data):
            import talib as ta  # TA-Lib for technical analysis
            
            data['RSI'] = ta.RSI(data['close'], timeperiod=14)
            data['MACD'], data['MACD_signal'], data['MACD_hist'] = ta.MACD(data['close'], fastperiod=12, slowperiod=26, signalperiod=9)
            data['Stoch_k'], data['Stoch_d'] = ta.STOCH(data['high'], data['low'], data['close'], fastk_period=14, slowk_period=3, slowd_period=3)
            
            return data

        def add_volume_indicators(data):
            import talib as ta  # TA-Lib for technical analysis
            
            data['OBV'] = ta.OBV(data['close'], data['volume'])
            return data

        def add_TC(data):
            window_size = 5
            data['mid_price'] = (data['high'] + data['low']) / 2
            data["mean_vol"] = data['mid_price'].pct_change().rolling(window=window_size).mean()
            data["mean_liq"] = data['volume'].rolling(window=window_size).mean()
            data = data.iloc[35:,:]
            
            # AC Calculation -> 
            x0 = 5000  # Initial number of shares to trade
            T = 1.0    # Total time horizon (e.g., 1 day)
            N = min(x0, 2400)  # Number of discrete time intervals
            eta = 0.0000001    # Temporary/permanent impact coefficient
            sigma = 0.02       # Volatility of the asset
            lambda_ = 0.1      # Risk aversion parameter

            # Time interval
            dt = 1

            def cost_function(x, eta, sigma, x0=10):
                x_cumsum = np.cumsum(x)
                x_half = x / 2
                temp_cost = np.sum(eta * (x**2))
                perm_cost = np.sum(eta * x * (x0 - x_cumsum + x_half))
                var_cost = np.sum(lambda_ * sigma**2 * (x**2))
                return (1 / (temp_cost + perm_cost + var_cost) / x0) * 10**(math.log10(x0) * 4 - 6)

            def almgren(row):
                x0 = 5000
                N = min(x0, 2400)
                eta = 1 / row['mean_liq'] if row['mean_liq'] != 0 else 0.0000001
                sigma = row['mean_vol']
                x_init = np.ones(N) * (x0 / N)
                return cost_function(x_init, eta, sigma, x0) / x0

            data['transaction_cost'] = data.apply(almgren, axis=1)
            return data

        def add_volatility_indicators(data):
            import talib as ta  # TA-Lib for technical analysis
            
            data['Upper_BB'], data['Middle_BB'], data['Lower_BB'] = ta.BBANDS(data['close'], timeperiod=20)
            data['ATR_1'] = ta.ATR(data['high'], data['low'], data['close'], timeperiod=1)
            data['ATR_2'] = ta.ATR(data['high'], data['low'], data['close'], timeperiod=2)
            data['ATR_5'] = ta.ATR(data['high'], data['low'], data['close'], timeperiod=5)
            data['ATR_10'] = ta.ATR(data['high'], data['low'], data['close'], timeperiod=10)
            data['ATR_20'] = ta.ATR(data['high'], data['low'], data['close'], timeperiod=20)
            return data

        def add_volatility(data, window=15):
            data['log_return'] = np.log(data['close'] / data['close'].shift(1))
            data['volatility'] = data['log_return'].rolling(window=window).std() * np.sqrt(window)
            return data

        def add_trend_indicators(data):
            import talib as ta  # TA-Lib for technical analysis
            
            data['ADX'] = ta.ADX(data['high'], data['low'], data['close'], timeperiod=14)
            data['+DI'] = ta.PLUS_DI(data['high'], data['low'], data['close'], timeperiod=14)
            data['-DI'] = ta.MINUS_DI(data['high'], data['low'], data['close'], timeperiod=14)
            data['CCI'] = ta.CCI(data['high'], data['low'], data['close'], timeperiod=5)
            
            return data

        def add_5_min_indicators(data):
            data['5_min_volatility'] = data['volatility'].transform(lambda x: x.rolling(window=5).std())
            data['5_min_volume'] = data['volume'].transform(lambda x: x.rolling(window=5).sum())
            data['5_min_TC'] = data['transaction_cost'].shift(5)
            return data

        def add_other_indicators(data):
            data['DLR'] = np.log(data['close'] / data['close'].shift(1))
            data['TWAP'] = data['close'].expanding().mean()
            data['VWAP'] = (data['volume'] * (data['high'] + data['low']) / 2).cumsum() / data['volume'].cumsum()
            data['market_liquidity'] = data['bid_size'] + data['ask_size']
            data['expected_price'] = data['bid_price']
            
            return data

        def add_all_indicators(data):
            data = add_momentum_indicators(data)
            data = add_volume_indicators(data)
            data = add_volatility_indicators(data)
            data = add_trend_indicators(data)
            data = add_other_indicators(data)
            data = add_volatility(data)
            data = add_TC(data)
            data = add_5_min_indicators(data)
            return data
        
        data_with_indicators = add_all_indicators(data)
        data_training = data_with_indicators.iloc[35:]  # Remove initial NaN values due to rolling calculations
        return data_training

    def add_forecasts(self, data):
        """
        Adds forecasts for various indicators to the data.

        Args:
        - data (pd.DataFrame): The DataFrame with technical indicators.

        Returns:
        - pd.DataFrame: The DataFrame with additional forecasted values.
        """
        print("LOGGING: Adding Forecasts to data...")
        
        from statsmodels.tsa.api import ARIMA, ExponentialSmoothing
        from joblib import Parallel, delayed

        def forecast_last_row(data, forecast_steps, columns, window_size):
            last_idx = len(data) - 1
            row_forecasts = {}
            for indicator, column in columns.items():
                steps, freq = forecast_steps[indicator]
                start_idx = max(0, last_idx - window_size)
                series = data[column].iloc[start_idx:last_idx+1]
                if len(series) < 2:
                    row_forecasts[f'forecast_{indicator}'] = None
                    continue
                if indicator in ['open', 'high', 'low', 'close', 'transaction_cost']:
                    model = ExponentialSmoothing(series, trend='add', seasonal=None)
                elif indicator == 'volatility':
                    model = ARIMA(series, order=(5, 1, 0))
                elif indicator == 'volume':
                    shift = 1 if series.min() <= 0 else 0
                    transformed_series = np.log(series + shift + 1)
                    model = ExponentialSmoothing(transformed_series, trend='add', seasonal=None)
                model_fit = model.fit()
                forecast_values = model_fit.forecast(steps=steps)
                if indicator == 'volume':
                    forecast_values = np.exp(forecast_values) - 1 - shift
                    forecast_values[forecast_values < 0] = 0  # Ensure non-negative values
                row_forecasts[f'forecast_6Hr_{indicator}'] = forecast_values.iloc[-1]
            return pd.Series(row_forecasts)

        forecast_steps = {
            'open': (360, '1T'),
            'high': (360, '1T'),
            'low': (360, '1T'),
            'close': (360, '1T'),
            'volatility': (360, '1T'),
            'volume': (360, '1T'),
            'transaction_cost': (360, '1T')
        }
        columns = {col: col for col in forecast_steps}
        last_row_forecast = forecast_last_row(data, forecast_steps, columns, 300)
        
        last_row_df = data.iloc[-1].to_frame().T.reset_index(drop=True)
        last_row_forecast_df = last_row_forecast.to_frame().T.reset_index(drop=True)
        input_row = pd.concat([last_row_df, last_row_forecast_df], axis=1)

        return input_row
    
    def add_real_time_forecasts(self, data, step):
        """
        Adds real-time forecasts for the next step in the trading process.

        Args:
        - data (pd.DataFrame): The DataFrame with technical indicators.
        - step (int): The forecast step indicating how far into the future to predict.

        Returns:
        - pd.DataFrame: The input row with real-time forecasts.
        """
        print("LOGGING: Adding Real-Time Forecasts to data...")
        
        from statsmodels.tsa.holtwinters import ExponentialSmoothing
        from statsmodels.tsa.arima.model import ARIMA

        def dynamic_forecast(data, step):
            """
            Generates a dynamic forecast for a set of indicators using historical data.

            Args:
            - data (pd.DataFrame): The historical market data.
            - step (int): The forecast step, indicating how far into the future the prediction should be made.

            Returns:
            - pd.DataFrame: A DataFrame containing the forecasted values for the last row in the dataset.
            """

            def forecast_last_row(data, forecast_steps, columns, window_size):
                last_idx = len(data) - 1
                row_forecasts = {'timestamp': data.index[last_idx]}
                state_forecasts = {}

                for indicator, column in columns.items():
                    steps, freq = forecast_steps[indicator]
                    start_idx = max(0, last_idx - window_size)
                    series = data[column].iloc[start_idx:last_idx+1]

                    # If insufficient data, skip forecast for this indicator
                    if len(series) < 2:
                        row_forecasts[f'forecast_{indicator}'] = None
                        continue

                    # Select appropriate model based on the indicator
                    if indicator in ['open', 'high', 'low', 'close', 'RSI', 'MACD',
                                   'MACD_signal', 'MACD_hist', 'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB',
                                   'Middle_BB', 'Lower_BB', 'ATR_1', 'ATR_2', 'ATR_5', 'ATR_10', 'ATR_20',
                                   'ADX', '+DI', '-DI', 'CCI']:
                        model = ExponentialSmoothing(series, trend='add', seasonal=None)
                    elif indicator == '5_min_TC' or indicator == 'transaction_cost':
                        model = ARIMA(series, order=(5, 1, 0))
#                     elif indicator == 'volume' or indicator == '5_min_volume':
#                         shift = 1 if series.min() <= 0 else 0
#                         transformed_series = np.log(series + shift + 1)
#                         model = ExponentialSmoothing(transformed_series, trend='add', seasonal=None)
                        
                    elif indicator == 'volume' or indicator == '5_min_volume':
                        shift = series.min() <= 0
                        transformed_series = np.log(series + shift + 1)
                        model = ExponentialSmoothing(transformed_series, trend='add', seasonal=None)
                    
                    elif indicator == 'volatility' or indicator == '5_min_volatility':
                        model = arch_model(series, mean='Constant', vol='Garch', p=1, q=1)
                        
                    
                    # Fit the model and generate forecast
                    if indicator == 'volatility' or indicator == '5_min_volatility':
                        model_fit = model.fit(disp='off')
                        forecast_values = model_fit.forecast(horizon=steps).variance.values[-1]
                    
                    elif indicator == 'volume' or indicator == '5_min_volume':
                        model_fit = model.fit()
                        forecast_values = model_fit.forecast(steps=steps)
                        forecast_values = np.exp(forecast_values) - 1 - shift
                        forecast_values[forecast_values < 0] = 1e-3  # Ensure small positive minimum
                        
                    else:
                        model_fit = model.fit()
                        forecast_values = model_fit.forecast(steps=steps)

                        
                        
#                         forecast_values = np.exp(forecast_values) - 1 - shift
#                         forecast_values[forecast_values < 0] = 0  # Ensure non-negative values
                    
                    if indicator == 'volume' or indicator == '5_min_volume':
                        state_forecasts[f'{indicator}'] = max(forecast_values.iloc[step-1], series.median())
                    elif indicator == 'volatility' or indicator == '5_min_volatility':
                        state_forecasts[f'{indicator}'] = np.sqrt(forecast_values[step-1])
                    else:
                        # Add forecast to state_forecast (for specific future step)
                        state_forecasts[f'{indicator}'] = forecast_values.iloc[step-1]

                    # Add future OHLCV to row_forecasts (e.g., forecast for the last step in 6 hours)
                    if indicator in ['open', 'high', 'low', 'close', 'transaction_cost']:
                        row_forecasts[f'forecast_6Hr_{indicator}'] = forecast_values.iloc[-1]
                    elif indicator in ['volume']:
                        row_forecasts[f'forecast_6Hr_{indicator}'] = max(forecast_values.iloc[-1], series.median())
                    elif indicator in ['volatility']:
                        row_forecasts[f'forecast_6Hr_{indicator}'] = np.sqrt(forecast_values[-1])

                # Create DataFrames for row-level and state forecasts
                row_forecasts_df = pd.DataFrame([row_forecasts]).reset_index(drop=True)
                state_forecasts_df = pd.DataFrame([state_forecasts]).reset_index(drop=True)

                return row_forecasts_df, state_forecasts_df

            forecast_steps = {
                'open': (step + 360, '1T'),
                'high': (step + 360, '1T'),
                'low': (step + 360, '1T'),
                'close': (step + 360, '1T'),
                'volume': (step + 360, '1T'),
                'volatility': (step + 360, '1T'),
                'transaction_cost': (step + 360, '1T'),
                'RSI': (step, '1T'),
                'MACD': (step, '1T'),
                'MACD_signal': (step, '1T'),
                'MACD_hist': (step, '1T'),
                'Stoch_k': (step, '1T'),
                'Stoch_d': (step, '1T'),
                'OBV': (step, '1T'),
                'Upper_BB': (step, '1T'),
                'Middle_BB': (step, '1T'),
                'Lower_BB': (step, '1T'),
                'ATR_1': (step, '1T'),
                'ADX': (step, '1T'),
                '+DI': (step, '1T'),
                '-DI': (step, '1T'),
                'CCI': (step, '1T'),
                '5_min_volatility': (step, '1T'),
                '5_min_volume': (step, '1T'),
                '5_min_TC': (step, '1T') # Add bid ask forecasts as well later
            }

            columns = {col: col for col in forecast_steps}
            last_row_forecast, data_forecast = forecast_last_row(data, forecast_steps, columns, 300)
            input_row = pd.concat([data_forecast, last_row_forecast], axis=1)
            return input_row

        input_row = dynamic_forecast(data, step)
        print(input_row)
        return input_row
    
    class CustomTradingEnvironment(gym.Env):
        """
        Custom Gym environment for simulating a trading scenario.
        """
        metadata = {'render.modes': ['human']}

        def __init__(self, data, scenario, action_space, preferred_timeframe=390, initial_inventory=10):
            """
            Initializes the Custom Trading Environment.

            Args:
            - data (pd.DataFrame): Current input data row.
            - scenario (dict): Contains any scenario-specific parameters.
            - action_space (gym.spaces): The action space for the agent.
            - preferred_timeframe (int, optional): The number of steps in which the trade should be completed (default is 390).
            - initial_inventory (int, optional): Initial inventory of shares to be traded (default is 10).
            """
            super(TradingModel.CustomTradingEnvironment, self).__init__()
            self.data = data        
            self.preferred_timeframe = preferred_timeframe
            self.initial_inventory = initial_inventory
            self.remaining_inventory = self.initial_inventory
            self.elapsed_time = 0
            self.trades = []
            self.timestamp = pd.to_datetime(self.data['timestamp'], unit='ms')

            # State columns contain market data and technical indicators
            self.state_columns = ['open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
                                  'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX', 
                                  '+DI', '-DI', 'CCI', 'transaction_cost', 'forecast_6Hr_open','forecast_6Hr_close',
                                  'forecast_6Hr_high','forecast_6Hr_low', 'forecast_6Hr_volatility', 
                                  'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost']

            # Define the action space and observation space for the environment
            self.action_space = action_space
            self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(len(self.state_columns),), dtype=np.float32)

        def reset(self):
            """
            Resets the environment to its initial state at the beginning of a new episode.

            Returns:
            - pd.Series: The last row of the data as the initial state.
            """
            print('------------------------------------------------Class resetted------------------------------------------------')
            self.remaining_inventory = self.initial_inventory
            self.elapsed_time = 0
            self.trades = []
            row_values = self.data.iloc[-1][self.state_columns].values
            state = self.data[self.state_columns].iloc[-1].values
            return state

        def get_next_valid_market_timestamp(self, current_timestamp, time_slice_minutes):
            """
            Calculates the next valid market timestamp based on the current timestamp and time slice.

            Args:
            - current_timestamp (datetime, pd.Timestamp, or numpy.int64): Current market timestamp.
            - time_slice_minutes (int): Minutes to add to the current timestamp.

            Returns:
            - datetime: The next valid market timestamp considering market hours, weekends, and holidays.
            """
            if isinstance(current_timestamp, (np.int64, int)):
                current_timestamp = datetime.utcfromtimestamp(current_timestamp)
            if isinstance(current_timestamp, (pd.Series, pd.DataFrame)):
                current_timestamp = current_timestamp.squeeze()
            if isinstance(current_timestamp, pd.Timestamp):
                current_timestamp = current_timestamp.to_pydatetime()

            market_start = current_timestamp.replace(hour=9, minute=30, second=0, microsecond=0)
            market_end = current_timestamp.replace(hour=16, minute=0, second=0, microsecond=0)
            new_timestamp = current_timestamp + timedelta(minutes=time_slice_minutes)

            if new_timestamp > market_end:
                new_timestamp = market_start + timedelta(days=1)
            if new_timestamp < market_start:
                new_timestamp = market_start

            while new_timestamp.weekday() >= 5:
                new_timestamp += timedelta(days=1)

            cal = USFederalHolidayCalendar()
            holidays = cal.holidays(start=new_timestamp, end=new_timestamp + timedelta(days=365)).to_pydatetime()
            while new_timestamp in holidays:
                new_timestamp += timedelta(days=1)
                new_timestamp = new_timestamp.replace(hour=9, minute=30, second=0, microsecond=0)

            return new_timestamp

        def step(self, action):
            """
            Executes a step in the environment by performing the action provided by the agent.

            Args:
            - action (np.array): An array where action[0] is the percentage of inventory to trade and action[1] is the timing of the next trade.

            Returns:
            - bool: Whether the episode is done (i.e., inventory is depleted or time is up).
            - dict: Additional information about the step taken.
            """
            action = self._add_noise_to_action(action)

            size_of_slice = action[0] * self.remaining_inventory
            size_of_slice = int(np.ceil(size_of_slice))
            timing_of_slice = int(np.ceil(action[1]))
#             print(f'timing_of_slice: {timing_of_slice}')

            self.elapsed_time += timing_of_slice

            if self.elapsed_time >= self.preferred_timeframe:
                size_of_slice = self.remaining_inventory

            self._take_action(size_of_slice)

            current_timestamp = self.timestamp
#             print(f"Current Timestamp: ", current_timestamp)
            next_timestamp = self.get_next_valid_market_timestamp(current_timestamp, timing_of_slice)

#             print(f'Current Timestamp: {current_timestamp}')
            self.elapsed_time += timing_of_slice
#             print(f'Next step: {next_timestamp}')

            done = self.remaining_inventory <= 0 or self.elapsed_time >= self.preferred_timeframe

            trade_info = {
                'timestamp': next_timestamp,
                'action': action,
                'shares': size_of_slice,
                'inventory': self.remaining_inventory,
                'time left': self.preferred_timeframe - self.elapsed_time
            }
            self.trades.append(trade_info)

            info = {'step': next_timestamp, 'action': action}
            self.timestamp = next_timestamp

            return done, info

        def _add_noise_to_action(self, action):
            """
            Adds noise to the agent's actions to simulate real-world uncertainties.

            Args:
            - action (np.array): An array containing the actions from the agent.

            Returns:
            - np.array: The action array with added noise.
            """
            noise_action_0 = np.random.normal(0, 0.02, size=action[0].shape)
            action[0] += noise_action_0
            action[0] = np.clip(action[0], self.action_space.low[0], self.action_space.high[0])

            noise_action_1 = np.random.normal(0, 5, size=action[1].shape)
            action[1] += noise_action_1
            action[1] = np.clip(action[1], self.action_space.low[1], self.action_space.high[1])

            return action

        def _take_action(self, size_of_slice):
            """
            Executes the trade by reducing the inventory based on the size of the slice.

            Args:
            - size_of_slice (int): The number of shares to trade.

            Returns:
            - None
            """
            self.remaining_inventory -= size_of_slice
#             print(f'Remaining inventory: {self.remaining_inventory}')
            if self.remaining_inventory < 0:
                self.remaining_inventory = 0

        def render(self, mode='human', close=False):
            """
            Renders the current state of the environment, including all trades executed so far.

            Args:
            - mode (str, optional): The mode of rendering (default is 'human').
            - close (bool, optional): Whether to close the render window (default is False).

            Returns:
            - None
            """
            print('--------------------------------------------------')
            return self.print_trades()

        def print_trades(self):
            """
            Prints a summary of all trades executed so far.

            Returns:
            - list: A list of dictionaries, each containing details of a trade.
            """
            trades_df = pd.DataFrame(self.trades)
            for trade in self.trades:
                print(f"Timestamp: {trade['timestamp']}, Action: {trade['action']}, Shares: {trade['shares']}, Inventory: {trade['inventory']}, TimeLeft: {trade['time left']}")

            return self.trades

    class MetaLearner:
        """
        This class is responsible for classifying the trading scenario based on transaction size.
        """
        def __init__(self):
            self.small_threshold = 10
            self.small_medium_threshold = 100
            self.medium_threshold = 500
            self.medium_large_threshold = 2000
            self.large_threshold = 10000

        def classify_scenario(self, transaction_size, timeframe=390):
            """
            Classifies the scenario based on the transaction size.

            Args:
            - transaction_size (int): The size of the transaction.
            - timeframe (int, optional): The timeframe for trade execution (default is 390 minutes).

            Returns:
            - str: The classified scenario ('small', 'small-medium', 'medium', 'medium-large', 'large').
            """
            if transaction_size < self.small_threshold:
                return 'small'
            elif self.small_threshold <= transaction_size < self.small_medium_threshold:
                return 'small-medium'
            elif self.small_medium_threshold <= transaction_size < self.medium_threshold:
                return 'medium'
            elif self.medium_threshold <= transaction_size < self.medium_large_threshold:
                return 'medium-large'
            elif self.medium_large_threshold <= transaction_size < self.large_threshold:
                return 'large'
            else:
                return 'large'
            
    class Model:
        """
        This class is responsible for loading pre-trained models for different trading scenarios.
        """
        def _load_model(self, env, filepath, policy_kwargs, best_hyperparameters):
            """
            Loads a pre-trained PPO model with a custom transformer policy from a specified file path.

            Args:
            - env (gym.Env): The trading environment for the model.
            - filepath (str): The file path to the saved model parameters.
            - policy_kwargs (dict): Additional keyword arguments for configuring the policy.
            - best_hyperparameters (dict): The best hyperparameters used for training the PPO model.

            Returns:
            - model (PPO): The loaded PPO model configured with the given environment and hyperparameters.
            """
            model = PPO(TradingModel.CustomTransformerPolicy, env, verbose=1, policy_kwargs=policy_kwargs, **best_hyperparameters)
            model.policy.load_state_dict(torch.load(filepath))
            print(f"Model loaded from {filepath}")
            return model

        def _get_small(self, data):
            """
            Configures and returns a PPO model for small trade scenarios.

            Args:
            - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

            Returns:
            - model (PPO): The PPO model configured for small trade scenarios.
            """
            file_path = 'Models/model_small.h5'
            action_space = spaces.Box(low=np.array([0.33, 30]), high=np.array([1, 50]), dtype=np.float32)
            scenario = 'small'
            env = TradingModel.CustomTradingEnvironment(data, scenario, action_space, preferred_timeframe=390, initial_inventory=10)
            
            best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                    'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
            model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
            return model

        def _get_small_medium(self, data):
            """
            Configures and returns a PPO model for small to medium trade scenarios.

            Args:
            - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

            Returns:
            - model (PPO): The PPO model configured for small to medium trade scenarios.
            """
            file_path = 'Models/model_small_med.h5'
            preferred_timeframe = 390
            initial_inventory = 100
            action_space = spaces.Box(low=np.array([0.2, 20]), high=np.array([0.66, 30]), dtype=np.float32)
            scenario = 'small-medium'
            env = TradingModel.CustomTradingEnvironment(data,scenario, action_space, preferred_timeframe=preferred_timeframe, initial_inventory=initial_inventory)
            
            best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                    'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
            model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
            return model

        def _get_medium(self, data):
            """
            Configures and returns a PPO model for medium trade scenarios.

            Args:
            - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

            Returns:
            - model (PPO): The PPO model configured for medium trade scenarios.
            """
            file_path = 'Models/model_med.h5'
            preferred_timeframe = 390
            initial_inventory = 500
            action_space = spaces.Box(low=np.array([0.20, 30]), high=np.array([0.50, 50]), dtype=np.float32)
            scenario = 'medium'
            env = TradingModel.CustomTradingEnvironment(data,scenario, action_space, preferred_timeframe=preferred_timeframe, initial_inventory=initial_inventory)

            best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                    'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
            model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
            return model

        def _get_medium_large(self, data):
            """
            Configures and returns a PPO model for medium to large trade scenarios.

            Args:
            - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

            Returns:
            - model (PPO): The PPO model configured for medium to large trade scenarios.
            """
            file_path = 'Models/model_med_lg.h5'
            preferred_timeframe = 390
            initial_inventory = 2000
            scenario = 'medium-large'
            action_space = spaces.Box(low=np.array([0.10, 30]), high=np.array([0.40, 50]), dtype=np.float32)
            env = TradingModel.CustomTradingEnvironment(data, scenario, action_space, preferred_timeframe=preferred_timeframe, initial_inventory=initial_inventory)

            best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                    'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
            model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
            return model

        def _get_large(self, data):
            """
            Configures and returns a PPO model for large trade scenarios.

            Args:
            - data (pd.DataFrame): Historical market data to be used in the trading environment - 1 row.

            Returns:
            - model (PPO): The PPO model configured for large trade scenarios.
            """
            file_path = 'Models/model_lg.h5'
            preferred_timeframe = 390
            initial_inventory = 10000
            action_space = spaces.Box(low=np.array([0.05, 30]), high=np.array([0.33, 50]), dtype=np.float32)
            scenario = 'large'
            env = TradingModel.CustomTradingEnvironment(data, scenario, action_space, preferred_timeframe=preferred_timeframe, initial_inventory=initial_inventory)
            
            best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                                    'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}
            model = self._load_model(env, file_path, {'scenario': scenario}, best_hyperparameters)
            return model

    class UNetTransformerEncoder(nn.Module):
        """
        Custom U-Net Transformer Encoder model for feature extraction.
        """
        def __init__(self, in_channels, out_channels, scenario='medium'):
            super(TradingModel.UNetTransformerEncoder, self).__init__()
            self.in_channels = in_channels
            self.out_channels = out_channels
            self.scenario = scenario

            # Define U-Net layers
            self.encoder1 = self._block(in_channels, 64)
            self.encoder2 = self._block(64, 128)
            self.encoder3 = self._block(128, 256)
            self.encoder4 = self._block(256, 512)
            self.bottleneck = self._block(512, 1024)
            self.decoder4 = self._block(1024 + 512, 512)
            self.decoder3 = self._block(512 + 256, 256)
            self.decoder2 = self._block(256 + 128, 128)
            self.decoder1 = self._block(128 + 64, out_channels)

            # Define Transformer layers based on scenario
            self.encoder_layer = self._get_encoder_layer(out_channels, scenario)
            self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=6)

        def _block(self, in_channels, out_channels):
            return nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(inplace=True),
                nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_channels),
                nn.ReLU(inplace=True)
            )

        def _get_encoder_layer(self, features_dim, scenario):
            if scenario == 'small':
                return nn.TransformerEncoderLayer(d_model=features_dim, nhead=4, dropout=0.2)
            elif scenario == 'small-medium':
                return nn.TransformerEncoderLayer(d_model=features_dim, nhead=8, dropout=0.15)
            elif scenario == 'medium':
                return nn.TransformerEncoderLayer(d_model=features_dim, nhead=8, dropout=0.1)
            elif scenario == 'medium-large':
                return nn.TransformerEncoderLayer(d_model=features_dim, nhead=16, dropout=0.08)
            elif scenario == 'large':
                return nn.TransformerEncoderLayer(d_model=features_dim, nhead=32, dropout=0.05)
            else:
                raise ValueError(f"Unknown scenario: {scenario}")

        def forward(self, x):
            enc1 = self.encoder1(x)
            enc2 = self.encoder2(F.max_pool1d(enc1, 2))
            enc3 = self.encoder3(F.max_pool1d(enc2, 2))
            enc4 = self.encoder4(F.max_pool1d(enc3, 2))

            bottleneck = self.bottleneck(F.max_pool1d(enc4, 2))

            dec4 = self.decoder4(torch.cat((F.interpolate(bottleneck, scale_factor=2), enc4), dim=1))
            dec3 = self.decoder3(torch.cat((F.interpolate(dec4, scale_factor=2), enc3), dim=1))
            dec2 = self.decoder2(torch.cat((F.interpolate(dec3, scale_factor=2), enc2), dim=1))
            dec1 = self.decoder1(torch.cat((F.interpolate(dec2, scale_factor=2), enc1), dim=1))

            # Transformer encoding
            x = self.transformer_encoder(dec1.permute(2, 0, 1)).permute(1, 2, 0)  # Permute for transformer encoder and back

            return x
        
    class CustomUNetTransformerModel(BaseFeaturesExtractor):
        """
        Custom U-Net Transformer model for feature extraction, used in the PPO policy.
        """
        def __init__(self, observation_space: spaces.Box, features_dim: int = 256, scenario: str = 'medium'):
            super(TradingModel.CustomUNetTransformerModel, self).__init__(observation_space, features_dim)
            self.embedding = nn.Linear(observation_space.shape[0], features_dim)  # Adapt the input size if necessary
            self.scenario = scenario
            self.unet_transformer = TradingModel.UNetTransformerEncoder(1, features_dim, scenario)

        def forward(self, observations: torch.Tensor) -> torch.Tensor:
            x = self.embedding(observations)
            x = x.unsqueeze(1)
            x = self.unet_transformer(x)
            x = x.mean(dim=2)
            return x
        
    class CustomTransformerPolicy(ActorCriticPolicy):
        """
        Custom Transformer Policy class for the PPO model.
        """
        def __init__(self, observation_space, action_space, lr_schedule, scenario='medium', *args, **kwargs):
            super(TradingModel.CustomTransformerPolicy, self).__init__(observation_space, action_space, lr_schedule, 
                                                          features_extractor_class=TradingModel.CustomUNetTransformerModel, 
                                                          features_extractor_kwargs={'features_dim': 256, 'scenario': scenario},
                                                          *args, **kwargs)

        
    def get_schedule(self, timeframe, transaction_size, input_row, data):
        """
        Generates a trading schedule based on the transaction size and input data.

        Args:
        - timeframe (int): The timeframe for trade execution.
        - transaction_size (int): The size of the transaction.
        - input_row (pd.DataFrame): The input data row with forecasts and technical indicators.

        Returns:
        - list: A list of trades executed based on the generated schedule.
        """
        print("LOGGING: Generating Schedule...")
        
        def infer_macro(timeframe, transaction_size, input_data, data):
            meta = TradingModel.MetaLearner()
            scenario = meta.classify_scenario(transaction_size, timeframe)
            print(f"Scenario: {scenario}")

            models = TradingModel.Model()

            if scenario == 'small':
                model = models._get_small(input_data)
            elif scenario == 'small-medium':
                model = models._get_small_medium(input_data)
            elif scenario == 'medium':
                model = models._get_medium(input_data)
            elif scenario == 'medium-large':
                model = models._get_medium_large(input_data)
            elif scenario == 'large':
                model = models._get_large(input_data)

            action_space = spaces.Box(low=np.array([0.10, 30]), high=np.array([0.40, 50]), dtype=np.float32)
            env = self.CustomTradingEnvironment(input_data, scenario, action_space, preferred_timeframe=timeframe, initial_inventory=transaction_size)
            obs = env.reset()
            obs = obs.astype(np.float32)
            done = False
            forecast_step = 0
            observations = []
            state_cols = ['open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
                                      'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX', 
                                      '+DI', '-DI', 'CCI', 'transaction_cost', 'forecast_6Hr_open','forecast_6Hr_close',
                                      'forecast_6Hr_high','forecast_6Hr_low', 'forecast_6Hr_volatility', 
                                      'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost']

            while not done:
                action, _states = model.predict(obs)
                step = int(np.ceil(action[1]))
                forecast_step += step
                obs = self.add_real_time_forecasts(data, forecast_step)
                obs = obs.squeeze()
                observations.append(obs)
                obs = obs[state_cols]
                done, info = env.step(action)

                if done:
                    break

            micro_input = pd.DataFrame(observations)
            trades = env.render()
            return trades, micro_input

        trades, micro_input = infer_macro(timeframe, transaction_size, input_row, data)
        return trades
            
    def run_pipeline(self):
        """
        Runs the entire pipeline for the trading model, including data retrieval, 
        technical indicator addition, forecasting, and generating trade schedules.

        Returns:
        - list: The list of trades generated by the model.
        """
        start_time = time.time()
        data = self.pull_data()
        data = self.add_technical_indicators(data)
        input_row = self.add_forecasts(data)
        trades = self.get_schedule(self.timeframe, self.inventory, input_row, data)
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"Execution time: {elapsed_time:.2f} seconds")
        
        return trades

In [63]:
model = TradingModel(ticker="AAPL", start_date="2024-08-01", end_date="2024-08-08", timeframe=500, inventory=10000)
trades = model.run_pipeline()

LOGGING: Pulling Data...
Merged data saved to merged_data_AAPL_2024-08-01_2024-08-08.csv
LOGGING: Adding Technical Indicators to data...
LOGGING: Adding Forecasts to data...
LOGGING: Generating Schedule...
Scenario: large
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Model loaded from Models/model_lg.h5
------------------------------------------------Class resetted------------------------------------------------
LOGGING: Adding Real-Time Forecasts to data...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


         open      high         low       close   volume  volatility  \
0  213.501528  213.4845  213.507541  213.490098  43806.0    0.001045   

   transaction_cost        RSI      MACD  MACD_signal  ...  5_min_volume  \
0          0.001014  45.734106 -0.091114    -0.075797  ...      268212.0   

   5_min_TC  timestamp  forecast_6Hr_open  forecast_6Hr_high  \
0  0.000676       5027         213.927278         213.718497   

   forecast_6Hr_low  forecast_6Hr_close  forecast_6Hr_volume  \
0        214.018032          213.750426              43806.0   

   forecast_6Hr_volatility  forecast_6Hr_transaction_cost  
0                 0.000893                       0.001007  

[1 rows x 33 columns]
LOGGING: Adding Real-Time Forecasts to data...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


         open     high         low       close   volume  volatility  \
0  213.537007  213.504  213.550082  213.511792  43806.0    0.000979   

   transaction_cost        RSI      MACD  MACD_signal  ...  5_min_volume  \
0          0.001007  44.381402 -0.178973      -0.1595  ...      268212.0   

   5_min_TC  timestamp  forecast_6Hr_open  forecast_6Hr_high  \
0   0.00068       5027         213.962757         213.737997   

   forecast_6Hr_low  forecast_6Hr_close  forecast_6Hr_volume  \
0        214.060573           213.77212              43806.0   

   forecast_6Hr_volatility  forecast_6Hr_transaction_cost  
0                 0.000893                       0.001007  

[1 rows x 33 columns]
LOGGING: Adding Real-Time Forecasts to data...


...

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


         open        high         low       close   volume  volatility  \
0  213.714403  213.601498  213.762787  213.620262  43806.0    0.000898   

   transaction_cost        RSI      MACD  MACD_signal  ...  5_min_volume  \
0          0.001007  37.617887 -0.618266    -0.578012  ...      268212.0   

   5_min_TC  timestamp  forecast_6Hr_open  forecast_6Hr_high  \
0   0.00068       5027         214.140152         213.835496   

   forecast_6Hr_low  forecast_6Hr_close  forecast_6Hr_volume  \
0        214.273278           213.88059              43806.0   

   forecast_6Hr_volatility  forecast_6Hr_transaction_cost  
0                 0.000893                       0.001007  

[1 rows x 33 columns]
LOGGING: Adding Real-Time Forecasts to data...


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(


         open        high         low       close   volume  volatility  \
0  213.749882  213.620998  213.805328  213.641956  43806.0    0.000896   

   transaction_cost        RSI      MACD  MACD_signal  ...  5_min_volume  \
0          0.001007  36.265183 -0.706125    -0.661714  ...      268212.0   

   5_min_TC  timestamp  forecast_6Hr_open  forecast_6Hr_high  \
0   0.00068       5027         214.175631         213.854996   

   forecast_6Hr_low  forecast_6Hr_close  forecast_6Hr_volume  \
0        214.315819          213.902284              43806.0   

   forecast_6Hr_volatility  forecast_6Hr_transaction_cost  
0                 0.000893                       0.001007  

[1 rows x 33 columns]
--------------------------------------------------
Timestamp: 2024-08-09 09:30:00, Action: [ 0.32113788 30.        ], Shares: 3212, Inventory: 6788, TimeLeft: 440
Timestamp: 2024-08-09 10:00:00, Action: [ 0.3248901 30.       ], Shares: 2206, Inventory: 4582, TimeLeft: 380
Timestamp: 2024-08-09 10

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/arch/univariate/base.py:766: ConvergenceWarning: The optimizer returned code 4. The message is:
Inequality constraints incompatible
See scipy.optimize.fmin_slsqp for code meaning.

  warnings.warn(
